# Data Loading and Fixed Split

This notebook is the canonical place to inspect the extracted dataset and create the fixed train / validation / test split used by every evaluation notebook.


## Run Safety

Before running this notebook, restart the kernel.

Why this matters:
- previous model runs can leave tensors and cached arrays in memory
- old variables can leak into the current run and make comparisons unfair
- we want every notebook to save a clean, reproducible result

Recommended workflow:
1. Restart kernel
2. Run all cells from top to bottom
3. Let the notebook save its outputs before closing it


In [1]:
# Memory cleanup before starting this notebook.
import gc

gc.collect()
try:
    import torch
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
except Exception:
    pass

print('Kernel memory cleanup complete. Start the notebook from here after a restart.')


Kernel memory cleanup complete. Start the notebook from here after a restart.


In [2]:
# Standard-library imports used throughout the evaluation notebooks.
import json
import math
import random
import sys
from collections import Counter
from dataclasses import dataclass
from functools import lru_cache
from pathlib import Path
from datetime import datetime

# Core scientific / ML stack.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset, Subset

# ROC / PR curves are useful for classification comparison if sklearn is available.
try:
    from sklearn.metrics import roc_curve, auc, precision_recall_curve, average_precision_score
    from sklearn.preprocessing import label_binarize
    SKLEARN_AVAILABLE = True
except Exception:
    SKLEARN_AVAILABLE = False

# Resolve the project root robustly no matter where Jupyter was launched from.
SEARCH_ROOT = Path.cwd().resolve()
PROJECT_ROOT = SEARCH_ROOT
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / '04_model_evaluation').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

EVAL_ROOT = PROJECT_ROOT / '04_model_evaluation'
THESIS_ROOT = EVAL_ROOT
NOTEBOOK_ROOT = EVAL_ROOT / 'notebooks'
RESULTS_ROOT = EVAL_ROOT / 'results'
WEIGHTS_ROOT = EVAL_ROOT / 'model_weights'
SPLITS_ROOT = EVAL_ROOT / 'splits'
PLOTS_ROOT = EVAL_ROOT / 'plots'
if str(NOTEBOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_ROOT))
ORIGINAL_DATASET_ROOT = PROJECT_ROOT / '03_dataset' / 'hybrid_maneuvers_dataset'
PREFERRED_DATASET_ROOT = ORIGINAL_DATASET_ROOT
DATASET_ROOT = PREFERRED_DATASET_ROOT

print('Project root:', PROJECT_ROOT)
print('Evaluation workspace:', THESIS_ROOT)
print('Original dataset root:', ORIGINAL_DATASET_ROOT)
print('Preferred dataset root:', PREFERRED_DATASET_ROOT)
print('Dataset root:', DATASET_ROOT)
print('Torch version:', torch.__version__)
print('Sklearn available for ROC/PR plots:', SKLEARN_AVAILABLE)


Project root: /home/basudeo/Documents/Thesis
Evaluation workspace: /home/basudeo/Documents/Thesis/04_model_evaluation
Original dataset root: /home/basudeo/Documents/Thesis/03_dataset/hybrid_maneuvers_dataset
Preferred dataset root: /home/basudeo/Documents/Thesis/03_dataset/hybrid_maneuvers_dataset
Dataset root: /home/basudeo/Documents/Thesis/03_dataset/hybrid_maneuvers_dataset
Torch version: 2.3.1+cu118
Sklearn available for ROC/PR plots: True


In [3]:
# Shared experiment configuration.
# Keep this block explicit and simple so the data pipeline stays easy to debug.
LABEL_MODE = 'reduced'      # Keep the 5-state mapping for teacher controller states.
SEED = 42
PAST_LEN = 10
FUTURE_LEN = 5             # Future steps used for ADE/FDE/RMSE trajectory targets.
SCAN_BEAMS = 512
RANGE_CLIP = 30.0
BATCH_SIZE = 32
EPOCHS = 30
EARLY_STOPPING_PATIENCE = 5
LR = 1e-3
WEIGHT_DECAY = 1e-4
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
MAX_SAMPLES = None
USE_CPU = False

CNN_HIDDEN = 96
GRAPH_HIDDEN = 96
FUSION_HIDDEN = 128
LSTM_HIDDEN = 128
LSTM_LAYERS = 1
MSG_PASSES = 2
DROPOUT = 0.10
TRANSFORMER_HEADS = 4
TRANSFORMER_LAYERS = 2
TRANSFORMER_FF = 256

device = torch.device('cuda' if torch.cuda.is_available() and not USE_CPU else 'cpu')
print('Device:', device)


Device: cuda


In [4]:
# Shared data and evaluation utilities live in one helper module so that
# all notebooks reuse the same dataset, split, path-remapping, and metric logic.
from dataset_helper import (
    build_label_mapping,
    build_sample_table,
    compute_trajectory_metrics,
    configure_helper,
    discover_frame_files,
    group_streams,
    load_npy_cached,
    remap_dataset_path,
    resample_scan,
    save_mean_step_error_plot,
    save_or_load_fixed_split,
    save_trajectory_overlay_plots,
    set_seed,
    timestamp_tag,
)

configure_helper(
    dataset_root=DATASET_ROOT,
    original_dataset_root=ORIGINAL_DATASET_ROOT,
    results_root=RESULTS_ROOT,
    weights_root=WEIGHTS_ROOT,
)


# Resolve the project structure relative to this notebook.
PROJECT_ROOT = Path.cwd().resolve().parent
EVAL_ROOT = PROJECT_ROOT
RESULTS_ROOT = EVAL_ROOT / 'results'
WEIGHTS_ROOT = EVAL_ROOT / 'model_weights'
SPLITS_ROOT = EVAL_ROOT / 'splits'

ORIGINAL_DATASET_ROOT = PROJECT_ROOT.parent / '03_dataset' / 'husky_control_dataset'
PREFERRED_DATASET_ROOT = ORIGINAL_DATASET_ROOT
DATASET_ROOT = PREFERRED_DATASET_ROOT

print('Original dataset root:', ORIGINAL_DATASET_ROOT)
print('Preferred dataset root:', PREFERRED_DATASET_ROOT)
print('Dataset root:', DATASET_ROOT)


In [5]:
labels, label_mapping = build_label_mapping(LABEL_MODE)
print('Controller-state metadata labels retained for analysis only:', labels)
print('Label mapping:', label_mapping)

frame_files = discover_frame_files(DATASET_ROOT)
assert frame_files, f'No frames.jsonl files found under {DATASET_ROOT}'
print('Found extracted datasets:')
for path in frame_files:
    print(' -', path)


Controller-state metadata labels retained for analysis only: ['go_to_goal', 'avoid_left', 'avoid_right', 'commit_forward', 'arrived']
Label mapping: {'bootstrap': None, 'go_to_goal': 'go_to_goal', 'avoid_left': 'avoid_left', 'avoid_right': 'avoid_right', 'commit_forward': 'commit_forward', 'reverse': None, 'recover': None, 'reassess': None, 'arrived': 'arrived', 'stop': None}
Found extracted datasets:
 - /home/basudeo/Documents/Thesis/03_dataset/husky_control_dataset/run_model_20260421_181846/frames.jsonl
 - /home/basudeo/Documents/Thesis/03_dataset/husky_control_dataset/run_model_20260421_185152/frames.jsonl
 - /home/basudeo/Documents/Thesis/03_dataset/husky_control_dataset/run_model_20260421_190059/frames.jsonl


In [6]:
streams = group_streams(DATASET_ROOT, allowed_labels=None, label_mapping=None)
print('Number of ego streams:', len(streams))

raw_label_counts = Counter()
for frames_path in frame_files:
    with frames_path.open() as f:
        for line in f:
            if not line.strip():
                continue
            row = json.loads(line)
            raw_label = str(row['teacher'].get('controller_state') or row['teacher'].get('label') or 'go_to_goal')
            raw_label_counts[raw_label] += 1

print('Raw controller-state counts:')
display(pd.DataFrame({'raw_label': list(raw_label_counts.keys()), 'count': list(raw_label_counts.values())}).sort_values('count', ascending=False))


Number of ego streams: 6
Raw controller-state counts:


,raw_label,count
0,go_to_goal,5623
6,arrived,1905
2,avoid_left,931
3,avoid_right,283
1,reassess,147
4,reverse,102
5,recover,95
7,bootstrap,10


In [7]:
# Peek at one raw frame and one raw scan so the new control dataset layout stays intuitive.
raw_frame = None
for frames_path in frame_files:
    with frames_path.open() as f:
        for line in f:
            if line.strip():
                raw_frame = json.loads(line)
                break
    if raw_frame is not None:
        break

print('Example frame keys:', sorted(raw_frame.keys()))
print('Example teacher block:')
print(json.dumps(raw_frame['teacher'], indent=2))
print('Example observation block:')
print(json.dumps(raw_frame['observation'], indent=2))
print('Example state block:')
print(json.dumps(raw_frame['state'], indent=2))
print('Example goal features:')
print(json.dumps(raw_frame['goal_features'], indent=2))

scan_ref = raw_frame['observation']['ego_planar_scan']
scan_arr = load_npy_cached(str(scan_ref['path']))
print('Raw planar scan shape:', scan_arr.shape)
print('First five rows of the raw scan array:')
print(scan_arr[:5])
print('Processed scan shape:', resample_scan(scan_arr, SCAN_BEAMS, RANGE_CLIP).shape)


Example frame keys: ['ego_id', 'episode_id', 'goal', 'goal_features', 'observation', 'other_husky', 'readiness', 'state', 'teacher', 'timestamp_ns', 'world_name']
Example teacher block:
{
  "command": {
    "linear_x": 1.1,
    "angular_z": 0.8236036056526855
  },
  "controller_state": "go_to_goal",
  "obstacle_action": "clear",
  "obstacle_clearance": null
}
Example observation block:
{
  "ego_planar_scan": {
    "path": "/home/basudeo/Documents/Thesis/03_dataset/husky_control_dataset/run_model_20260421_181846/assets/husky_local/planar_scan/1776788416835812195.npy",
    "timestamp_ns": 1776788416835812195,
    "modality": "planar_scan",
    "shape": [
      920,
      2
    ],
    "dtype": "float32"
  },
  "ego_front_pointcloud": null
}
Example state block:
{
  "x": 0.10877938491397057,
  "y": 0.004437023459024164,
  "z": 0.0,
  "qx": 0.0,
  "qy": 0.0,
  "qz": 0.040755371719583405,
  "qw": 0.9991691546860314,
  "yaw": 0.08153332521720563,
  "vx": 1.100000000000003,
  "vy": 0.0,
  "vz"

# Build and Save the Fixed Split

Every model notebook should evaluate on the exact same train / validation / test partition. This notebook creates the canonical split file used by the rest of the evaluation suite.


In [8]:
sample_table = build_sample_table(streams, past_len=PAST_LEN, future_len=FUTURE_LEN)
split_path = SPLITS_ROOT / f'husky_control_split_{LABEL_MODE}_seed{SEED}_past{PAST_LEN}_future{FUTURE_LEN}.json'
split_info = save_or_load_fixed_split(
    sample_table=sample_table,
    split_path=split_path,
    seed=SEED,
    train_ratio=TRAIN_RATIO,
    val_ratio=VAL_RATIO,
    past_len=PAST_LEN,
    future_len=FUTURE_LEN,
)

assert split_info['sample_count'] == len(sample_table), (
    f"Split file {split_path} does not match the current dataset: "
    f"split has {split_info['sample_count']} samples, current table has {len(sample_table)}"
)
print('Split file:', split_path)
print('Total windowed samples:', split_info['sample_count'])
print('Future horizon:', split_info['future_len'])
print('Train samples:', len(split_info['train_indices']))
print('Validation samples:', len(split_info['val_indices']))
print('Test samples:', len(split_info['test_indices']))


Split file: /home/basudeo/Documents/Thesis/04_model_evaluation/splits/husky_control_split_reduced_seed42_past10_future5.json
Total windowed samples: 9012
Future horizon: 5
Train samples: 6308
Validation samples: 1351
Test samples: 1353


In [9]:
# Show a quick table of sample ids from each split for sanity.
split_preview = pd.DataFrame({
    'train_sample_id': [split_info['sample_ids'][idx] for idx in split_info['train_indices'][:5]],
    'val_sample_id': [split_info['sample_ids'][idx] for idx in split_info['val_indices'][:5]],
    'test_sample_id': [split_info['sample_ids'][idx] for idx in split_info['test_indices'][:5]],
})
display(split_preview)


,train_sample_id,val_sample_id,test_sample_id
0,stream002_start00303,stream003_start01205,stream004_start01440
1,stream003_start01056,stream004_start00518,stream003_start01051
2,stream003_start01153,stream001_start00930,stream002_start00460
3,stream003_start00957,stream000_start00922,stream000_start00884
4,stream003_start00142,stream000_start01579,stream003_start00836


In [10]:
# Inspect one trajectory target so future-metric supervision stays easy to understand.
preview_meta = sample_table[0]
print('Preview sample id:', preview_meta['sample_id'])
print('Preview label:', preview_meta['label'])
print('Future relative XY target shape:', np.asarray(preview_meta['future_xy']).shape)
display(pd.DataFrame({
    'dt_seconds': preview_meta['future_dt'],
    'future_dx': [xy[0] for xy in preview_meta['future_xy']],
    'future_dy': [xy[1] for xy in preview_meta['future_xy']],
}))


Preview sample id: stream000_start00000
Preview label: go_to_goal
Future relative XY target shape: (5, 2)


,dt_seconds,future_dx,future_dy
0,0.322798,0.149526,0.056867
1,0.639647,0.295441,0.122456
2,0.949294,0.437638,0.195767
3,1.262271,0.576086,0.275940
4,1.569809,89.084064,15.140346
